# YT SEO Insights Generator
### AI-Powered YouTube SEO Analysis

> **Stack:** Groq llama-3.3-70b-versatile | YouTube scraping | MLflow | Streamlit  
> **Colab-ready:** All secrets from Colab Secrets (lock icon in sidebar)

---

## Cell 1 - Project Configuration
All paths, audio settings, and MLflow experiment name.

In [ ]:
import os

# ── Hardcoded Colab paths ─────────────────────────────────────────────────────
AUDIO_DIR   = '/content/audio'      # <- upload your audio tracks here
IMAGE_DIR   = '/content/images'     # <- upload your cover image here
OUTPUT_DIR  = '/content/output'     # <- all generated files land here

# Allowed audio file extensions for the mixtape builder
ALLOWED_AUDIO_EXT = ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg')

# ── Mixtape settings ──────────────────────────────────────────────────────────
TRANSITION_MS  = 6000           # crossfade duration in milliseconds
OUTPUT_MP3     = 'mixtape.mp3'
OUTPUT_MP4     = 'mixtape_vid.mp4'
MIXTAPE_NAME   = 'Afro House Summer Mix'
GENRE          = 'Afro House'
COVER_IMAGE    = 'cover.jpg'    # filename inside IMAGE_DIR

# ── MLflow experiment ─────────────────────────────────────────────────────────
MLFLOW_EXPERIMENT = 'youtube-mixtape-automation'

# ── Create required runtime directories ──────────────────────────────────────
for d in [AUDIO_DIR, IMAGE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Config loaded.')
print(f'  Audio dir  : {AUDIO_DIR}')
print(f'  Image dir  : {IMAGE_DIR}')
print(f'  Output dir : {OUTPUT_DIR}')
print(f'  Transition : {TRANSITION_MS} ms')


## Cell 2 - Install Dependencies

In [ ]:
# Install all runtime dependencies.
# streamlit     : web UI framework
# groq          : Groq API client (replaces openai)
# python-dotenv : loads .env files (parity with local dev)
# requests      : HTTP scraping of YouTube pages
# mlflow        : experiment tracking and artifact logging
# pyngrok       : exposes local Streamlit port to a public URL

%pip install -q streamlit groq python-dotenv requests mlflow pyngrok
print('All dependencies installed.')


## Cell 3 - Load API Keys from Colab Secrets
Keys are read from the **Colab Secret Store** (lock icon in left sidebar).  
Add `GROQ_API_KEY` there before running this cell.

In [ ]:
from google.colab import userdata  # Colab built-in secret manager

# ── Read Groq API key from Colab Secrets ──────────────────────────────────────
# Sidebar -> Secrets (lock icon) -> + Add new secret
# Name : GROQ_API_KEY   Value: gsk_...
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

if not GROQ_API_KEY:
    raise ValueError(
        'GROQ_API_KEY not found in Colab Secrets. '
        'Please add it via the lock icon in the left sidebar.'
    )

# Inject into environment so the Groq SDK picks it up automatically
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print('GROQ_API_KEY loaded from Colab Secrets.')


## Cell 4 - Logger (src/common/logger.py)
File-based logger that writes timestamped logs to `logs/`.  
Every module obtains its logger via `get_logger(__name__)`.

In [ ]:
import os
import logging
from datetime import datetime

# ── Logging directory ─────────────────────────────────────────────────────────
LOGS_DIR = 'logs'
os.makedirs(LOGS_DIR, exist_ok=True)

# ── Log file: one file per session named with current timestamp ───────────────
LOG_FILE = os.path.join(
    LOGS_DIR,
    f"log_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.log"
)

# ── Root logger configuration ─────────────────────────────────────────────────
logging.basicConfig(
    filename=LOG_FILE,
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)

def get_logger(name: str) -> logging.Logger:
    """Return a named logger that inherits the root configuration."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    return logger

print(f'Logger initialised. Log file -> {LOG_FILE}')


## Cell 5 - Custom Exception (src/common/custom_exception.py)
Enriches every exception with the **file name** and **line number** where it originated, making debugging easier in deep call stacks.

In [ ]:
import sys

class CustomException(Exception):
    """
    Project-wide exception class.

    Enriches any exception with:
      - original message
      - root-cause exception (if provided)
      - source file name and line number at the raise point
    """

    def __init__(self, message: str, error_detail=None):
        # Build the enriched message and store it
        self.error_message = self._build_message(message, error_detail)
        super().__init__(self.error_message)

    @staticmethod
    def _build_message(message: str, error_detail) -> str:
        """Construct a detailed error string using the active traceback."""
        _, _, exc_tb = sys.exc_info()

        # Guard: exc_tb is None when called outside an except block
        if exc_tb is not None:
            file_name   = exc_tb.tb_frame.f_code.co_filename
            line_number = exc_tb.tb_lineno
        else:
            file_name   = 'unknown'
            line_number = 'unknown'

        return (
            f"{message} | "
            f"Error: {error_detail} | "
            f"File: {file_name} | "
            f"Line: {line_number}"
        )

    def __str__(self) -> str:
        return self.error_message

print('CustomException class defined.')


## Cell 6 - Video Extractor (src/utils/video_extractor.py)
Scrapes YouTube metadata (title, author, views, duration, thumbnail) directly from the page HTML. No YouTube Data API key required.

Supported URL formats:
- `https://www.youtube.com/watch?v=VIDEO_ID`
- `https://youtu.be/VIDEO_ID`
- `https://www.youtube.com/shorts/VIDEO_ID`

In [ ]:
import re
import requests

class VideoExtractor:
    """
    Extracts YouTube video metadata by scraping the watch-page HTML.

    No YouTube Data API key needed; metadata is parsed from the
    embedded JSON-LD and Open Graph tags on every watch page.
    """

    def __init__(self):
        self.logger = get_logger(__name__)
        self.logger.info('VideoExtractor initialised.')

    # ── Private helpers ───────────────────────────────────────────────────────

    def _extract_video_id(self, url: str) -> str:
        """
        Parse a YouTube URL and return the 11-character video ID.

        Supports standard watch links, youtu.be, and YouTube Shorts.

        Raises:
            CustomException: if URL is empty, not a string, or
                             has no recognisable video ID.
        """
        if not url or not isinstance(url, str):
            raise CustomException('URL is empty or not a string.')

        # Three regex patterns covering the most common YouTube URL formats
        patterns = [
            r'(?:v=|\/)([0-9A-Za-z_-]{11})',           # ?v= and /VIDEO_ID paths
            r'youtu\.be\/([0-9A-Za-z_-]{11})',          # shortened links
            r'youtube\.com\/shorts\/([0-9A-Za-z_-]{11})'  # Shorts
        ]

        for pattern in patterns:
            match = re.search(pattern, url)
            if match:
                video_id = match.group(1)
                self.logger.info(f'Extracted video ID: {video_id}')
                return video_id

        raise CustomException('Could not extract a valid 11-char video ID from the URL.')

    def _fetch_youtube_metadata(self, video_id: str) -> dict:
        """
        Fetch the YouTube watch page and extract metadata via regex.

        Returns dict with keys:
            title, thumbnail_url, duration (int seconds),
            views (int), author, platform, video_id.

        Raises:
            CustomException: on HTTP error or parse failure.
        """
        self.logger.info(f'Fetching metadata for video_id={video_id}')

        watch_url = f'https://www.youtube.com/watch?v={video_id}'
        headers   = {'User-Agent': 'Mozilla/5.0'}  # mimic a real browser

        response = requests.get(watch_url, headers=headers, timeout=10)

        if response.status_code != 200:
            self.logger.error(f'HTTP {response.status_code} for {video_id}')
            raise CustomException(f'HTTP request failed (status {response.status_code}).')

        html = response.text

        def _re(pattern: str, default: str = '') -> str:
            """Return first capture group from regex or default."""
            m = re.search(pattern, html)
            return m.group(1) if m else default

        # ── Extract individual metadata fields ────────────────────────────────
        title     = _re(r'<meta property="og:title" content="([^"]+)"', 'Untitled Video')
        thumbnail = f'https://img.youtube.com/vi/{video_id}/hqdefault.jpg'
        duration  = int(_re(r'"lengthSeconds":"(\d+)"', '400'))  # seconds
        views     = int(_re(r'"viewCount":"(\d+)"',     '100'))
        author    = _re(r'"author":"([^"]+)"',           'Unknown Creator')

        metadata = {
            'title'        : title,
            'thumbnail_url': thumbnail,
            'duration'     : duration,
            'views'        : views,
            'author'       : author,
            'platform'     : 'YouTube',
            'video_id'     : video_id,
        }

        self.logger.info('Metadata extracted successfully.')
        return metadata

    # ── Public API ────────────────────────────────────────────────────────────

    def get_video_metadata(self, url: str) -> dict:
        """Parse URL, fetch and return metadata dict."""
        try:
            self.logger.info(f'Processing URL: {url}')
            video_id = self._extract_video_id(url)
            return self._fetch_youtube_metadata(video_id)
        except Exception as e:
            self.logger.error(f'Error processing URL: {e}')
            raise CustomException('Error while processing the YouTube URL.', e)


# ── Module-level convenience function ─────────────────────────────────────────

def get_video_metadata(url: str) -> dict:
    """
    Shortcut: create VideoExtractor and return metadata for url.

    Example:
        meta = get_video_metadata('https://www.youtube.com/watch?v=dQw4w9WgXcQ')
    """
    return VideoExtractor().get_video_metadata(url)

print('VideoExtractor class defined.')


## Cell 7 - SEO Engine (src/core/seo_engine.py)
Sends video metadata to **Groq (llama-3.3-70b-versatile)** and parses structured JSON into:

| Key | Content |
|---|---|
| `tags` | 35 SEO hashtags |
| `audience` | Target audience paragraph |
| `timestamps` | Auto-generated chapter markers |
| `flaws` | 2-3 SEO issues with fixes |

In [ ]:
import json
from groq import Groq

class SEOEngine:
    """
    Generates YouTube SEO insights from video metadata using Groq.

    Model used: llama-3.3-70b-versatile (fast, free-tier friendly).

    Workflow:
        1. _build_prompt()    - construct the structured JSON prompt
        2. Groq API call      - send prompt, receive raw text
        3. _parse_json()      - extract valid JSON from the response
        4. _validate_output() - assert all required keys are present
    """

    # Groq model to use — swap here if you want a different model
    MODEL = 'llama-3.3-70b-versatile'

    def __init__(self):
        self.logger = get_logger(__name__)
        self.logger.info('SEOEngine initialised.')

        # Guard: ensure API key is present before creating the client
        if not os.environ.get('GROQ_API_KEY'):
            raise CustomException(
                'GROQ_API_KEY not found. Run Cell 3 (Secrets) first.'
            )

        try:
            # Groq SDK picks up GROQ_API_KEY from the environment automatically
            self.client = Groq(api_key=os.environ['GROQ_API_KEY'])
            self.logger.info('Groq client connected.')
        except Exception as e:
            self.logger.error('Failed to initialise Groq client.')
            raise CustomException('Failed to initialise Groq client.', e)

    # ── Private helpers ───────────────────────────────────────────────────────

    def _build_prompt(self, metadata: dict) -> str:
        """
        Construct the prompt from video metadata.

        Timestamp count is scaled by duration:
            num_timestamps = clamp(duration_min / 2, 5, 15)

        Args:
            metadata: dict returned by VideoExtractor.

        Returns:
            Prompt string ready for the model.
        """
        try:
            title    = metadata['title']
            duration = metadata['duration']    # seconds
            platform = metadata['platform']

            minutes        = duration // 60
            # Clamp timestamp count between 5 and 15
            num_timestamps = min(15, max(5, int(minutes / 2)))

            prompt = (
                'You MUST respond with VALID JSON ONLY. '
                'No preamble, no markdown fences.\n\n'
                f'Video details:\n'
                f'  Title    : "{title}"\n'
                f'  Platform : {platform}\n'
                f'  Duration : {duration} seconds\n\n'
                'Return JSON exactly in this structure:\n\n'
                '{\n'
                '  "tags": ["tag1", ..., "tag35"],\n'
                '  "audience": "One paragraph describing the ideal target audience.",\n'
                '  "timestamps": [\n'
                '    {"time": "00:00", "description": "Intro"},\n'
                '    ...\n'
                '  ],\n'
                '  "flaws": [\n'
                '    {\n'
                '      "issue"        : "Specific SEO problem found in this video.",\n'
                '      "why_it_hurts" : "Why this lowers ranking or watch-time.",\n'
                '      "fix"          : "Clear, actionable improvement step."\n'
                '    },\n'
                '    ...\n'
                '  ]\n'
                '}\n\n'
                'Hard rules:\n'
                '- Produce EXACTLY 35 tags.\n'
                f'- Produce EXACTLY {num_timestamps} timestamps spread across the full duration.\n'
                '- Produce 2-3 flaws.\n'
                '- All content must be in English.\n'
            )
            return prompt

        except Exception as e:
            self.logger.error('Error building prompt.')
            raise CustomException('Error building prompt.', e)

    def _parse_json(self, raw_output: str) -> dict:
        """
        Parse the model's raw text into a Python dict.

        Strategy 1: direct json.loads().
        Strategy 2 (fallback): slice out the first complete {...} block,
        handling cases where the model adds markdown fences.

        Raises:
            CustomException: if neither strategy succeeds.
        """
        try:
            return json.loads(raw_output)
        except Exception:
            try:
                # Slice out the outermost JSON object
                start = raw_output.find('{')
                end   = raw_output.rfind('}') + 1
                return json.loads(raw_output[start:end])
            except Exception as e:
                raise CustomException('Failed to parse JSON from model output.', e)

    def _validate_output(self, data: dict) -> None:
        """Assert all four required keys are present in the parsed output."""
        for key in ['tags', 'audience', 'timestamps', 'flaws']:
            if key not in data:
                self.logger.error(f"Model output missing key: '{key}'")
                raise CustomException(f"Model output missing required key: '{key}'")

    # ── Public API ────────────────────────────────────────────────────────────

    def generate(self, video_metadata: dict) -> dict:
        """
        Full pipeline: prompt -> Groq API call -> parse -> validate -> return.

        Args:
            video_metadata: dict from VideoExtractor.get_video_metadata().

        Returns:
            dict with keys: tags, audience, timestamps, flaws.
        """
        try:
            self.logger.info('Starting SEO insights generation.')

            prompt = self._build_prompt(video_metadata)

            # ── Groq inference call ───────────────────────────────────────────
            # The Groq SDK mirrors the OpenAI chat completions interface,
            # so only the client class and model name differ from OpenAI.
            response = self.client.chat.completions.create(
                model=self.MODEL,
                temperature=0.4,   # low = deterministic, structured output
                messages=[
                    {'role': 'system', 'content': 'Return ONLY valid JSON. No extra text.'},
                    {'role': 'user',   'content': prompt}
                ]
            )

            raw = response.choices[0].message.content.strip()
            self.logger.info('Raw model output received from Groq.')

            data = self._parse_json(raw)
            self._validate_output(data)

            self.logger.info('SEO insights generated successfully.')
            return data

        except Exception as e:
            self.logger.error(f'SEO generation failed: {e}')
            raise CustomException('Error during SEO insights generation.', e)

print('SEOEngine class defined (Groq backend).')


## Cell 8 - MLflow Experiment Tracking
Logs each run as an MLflow experiment:
- **Params:** video title, duration, platform
- **Metrics:** tag count, timestamp count, flaw count
- **Artifact:** full `seo_results.json`

In [ ]:
import mlflow

# ── Point MLflow at a local directory inside OUTPUT_DIR ───────────────────────
# All run data (params, metrics, artifacts) will be stored there.
mlflow.set_tracking_uri(f'file://{OUTPUT_DIR}/mlruns')
mlflow.set_experiment(MLFLOW_EXPERIMENT)

def log_run_to_mlflow(metadata: dict, seo_data: dict) -> None:
    """
    Log one SEO generation run to MLflow.

    Logs:
        params   - video-level identifiers (title, platform, duration)
        metrics  - output counts (tags, timestamps, flaws)
        artifact - full seo_data dict as seo_results.json
    """
    with mlflow.start_run():
        # ── Parameters ────────────────────────────────────────────────────────
        mlflow.log_param('video_title',   metadata.get('title',    'unknown'))
        mlflow.log_param('platform',      metadata.get('platform', 'YouTube'))
        mlflow.log_param('duration_sec',  metadata.get('duration', 0))
        mlflow.log_param('author',        metadata.get('author',   'unknown'))

        # ── Metrics ───────────────────────────────────────────────────────────
        mlflow.log_metric('tag_count',       len(seo_data.get('tags',       [])))
        mlflow.log_metric('timestamp_count', len(seo_data.get('timestamps', [])))
        mlflow.log_metric('flaw_count',      len(seo_data.get('flaws',      [])))

        # ── Artifact: full JSON result file ───────────────────────────────────
        artifact_path = os.path.join(OUTPUT_DIR, 'seo_results.json')
        with open(artifact_path, 'w') as f:
            json.dump(seo_data, f, indent=2)
        mlflow.log_artifact(artifact_path)

    print(f'MLflow run logged. Artifact -> {artifact_path}')

print(f"MLflow configured. Experiment: '{MLFLOW_EXPERIMENT}'")


## Cell 9 - Run the Full Pipeline
Replace `YOUTUBE_URL` with any public YouTube link, then run this cell to:
1. Scrape video metadata
2. Generate SEO insights via GPT-4o-mini
3. Log the run to MLflow
4. Display all results inline

In [ ]:
# ── Input: replace with any public YouTube URL ───────────────────────────────
YOUTUBE_URL = 'https://www.youtube.com/watch?v=dQw4w9WgXcQ'   # <- replace me

# ── Step 1: Extract video metadata ───────────────────────────────────────────
print('Fetching video metadata...')
metadata = get_video_metadata(YOUTUBE_URL)

print(f"  Title    : {metadata['title']}")
print(f"  Author   : {metadata['author']}")
print(f"  Views    : {metadata['views']:,}")
print(f"  Duration : {metadata['duration'] // 60} min {metadata['duration'] % 60} sec")

# ── Step 2: Generate SEO insights via Groq (llama-3.3-70b-versatile) ──────────
print('\nGenerating SEO insights...')
engine   = SEOEngine()
seo_data = engine.generate(metadata)

# ── Step 3: Log run to MLflow ─────────────────────────────────────────────────
print('\nLogging run to MLflow...')
log_run_to_mlflow(metadata, seo_data)

# ── Step 4: Display all results inline ───────────────────────────────────────
sep = '=' * 60

print(f'\n{sep}')
print('SEO-FRIENDLY TAGS (35)')
print(sep)
print('  '.join(f'#{t}' for t in seo_data['tags']))

print(f'\n{sep}')
print('TARGET AUDIENCE ANALYSIS')
print(sep)
print(seo_data['audience'])

print(f'\n{sep}')
print('AI-GENERATED TIMESTAMPS')
print(sep)
for ts in seo_data['timestamps']:
    print(f"  {ts['time']}  -  {ts['description']}")

print(f'\n{sep}')
print('FLAWS AND IMPROVEMENT SUGGESTIONS')
print(sep)
for i, flaw in enumerate(seo_data['flaws'], 1):
    print(f"\n  [{i}] Issue        : {flaw['issue']}")
    print(f"       Why it hurts : {flaw['why_it_hurts']}")
    print(f"       Fix          : {flaw['fix']}")

print('\nPipeline complete.')


## Cell 10 - Save Outputs and Checkpoint
Saves four artifact files to `OUTPUT_DIR`:
- `seo_results.json` — full structured output (also logged by MLflow)
- `seo_tags.txt` — hashtag list ready to paste into YouTube
- `seo_timestamps.txt` — chapter list for the video description
- `seo_flaws.txt` — flaw report

In [ ]:
# ── Save full JSON results ────────────────────────────────────────────────────
results_json_path = os.path.join(OUTPUT_DIR, 'seo_results.json')
with open(results_json_path, 'w') as f:
    json.dump(seo_data, f, indent=2)
print(f'Saved full JSON    -> {results_json_path}')

# ── Save hashtag list (copy-paste ready for YouTube description) ──────────────
tags_txt_path = os.path.join(OUTPUT_DIR, 'seo_tags.txt')
with open(tags_txt_path, 'w') as f:
    f.write(' '.join(f'#{t}' for t in seo_data['tags']))
print(f'Saved tags list    -> {tags_txt_path}')

# ── Save timestamp list (copy-paste ready for YouTube description) ────────────
timestamps_txt_path = os.path.join(OUTPUT_DIR, 'seo_timestamps.txt')
with open(timestamps_txt_path, 'w') as f:
    for ts in seo_data['timestamps']:
        f.write(f"{ts['time']} {ts['description']}\n")
print(f'Saved timestamps   -> {timestamps_txt_path}')

# ── Save flaws / improvement report ──────────────────────────────────────────
flaws_txt_path = os.path.join(OUTPUT_DIR, 'seo_flaws.txt')
with open(flaws_txt_path, 'w') as f:
    for i, flaw in enumerate(seo_data['flaws'], 1):
        f.write(f"[{i}] Issue        : {flaw['issue']}\n")
        f.write(f"     Why it hurts : {flaw['why_it_hurts']}\n")
        f.write(f"     Fix          : {flaw['fix']}\n\n")
print(f'Saved flaws report -> {flaws_txt_path}')

print(f'\nAll outputs saved to: {OUTPUT_DIR}')


## Cell 11 - Launch Streamlit App (Optional)
Writes `app.py` and exposes it via **ngrok** so you can use the full web UI
from any browser while the Colab session is active.

**Prerequisite:** add `NGROK_AUTHTOKEN` to Colab Secrets (free at ngrok.com).

In [ ]:
import subprocess
import time
from pyngrok import ngrok
from google.colab import userdata  # type: ignore

# ── Write app.py to the Colab working directory ───────────────────────────────
# The file mirrors the original app.py from the project repo.
# It imports from the src/ package; make sure to run the project in the
# correct working directory (or add src/ to sys.path first).
APP_LINES = [
    'import os\n',
    'from dotenv import load_dotenv\n',
    'import streamlit as st\n',
    'from src.utils.video_extractor import get_video_metadata\n',
    'from src.core.seo_engine import SEOEngine\n',
    'from src.common.logger import get_logger\n',
    '\n',
    'load_dotenv()\n',
    'logger = get_logger(__name__)\n',
    '\n',
    'st.set_page_config(page_title="YT SEO Insights", layout="wide")\n',
    'st.title("AI YouTube SEO Insights Generator")\n',
    'st.write("AI-generated Tags | Audience | Timestamps | Flaws")\n',
    '\n',
    'with st.sidebar:\n',
    '    st.header("API Settings")\n',
    '    api_key = st.text_input("Groq API Key", type="password")\n',
    '    if api_key:\n',
    '        os.environ["GROQ_API_KEY"] = api_key\n',
    '        logger.info("API key set from sidebar.")\n',
    '\n',
    'st.divider()\n',
    'url = st.text_input("Enter YouTube URL",\n',
    '                     placeholder="https://www.youtube.com/watch?v=...")\n',
    '\n',
    'if "seo_data" not in st.session_state:\n',
    '    st.session_state.seo_data = None\n',
    '\n',
    'if url:\n',
    '    try:\n',
    '        metadata = get_video_metadata(url)\n',
    '        st.subheader("Video Details")\n',
    '        st.write(f"**Title:** {metadata[chr(39)]title[chr(39)]}")\n',
    '        st.write(f"**Creator:** {metadata[chr(39)]author[chr(39)]}")\n',
    '        st.write(f"**Views:** {metadata[chr(39)]views[chr(39)]:,}")\n',
    '        st.write(f"**Duration:** {metadata[chr(39)]duration[chr(39)] // 60} min")\n',
    '        st.image(metadata["thumbnail_url"], width=400)\n',
    '        if st.button("Generate SEO Insights"):\n',
    '            if not os.getenv("GROQ_API_KEY"):\n',
    '                st.error("Add your API key in the sidebar first.")\n',
    '            else:\n',
    '                with st.spinner("Analysing with AI..."):\n',
    '                    seo = SEOEngine()\n',
    '                    st.session_state.seo_data = seo.generate(metadata)\n',
    '    except Exception as e:\n',
    '        logger.error(f"Error: {e}")\n',
    '        st.error("Unexpected error. Check logs for details.")\n',
    '\n',
    'data = st.session_state.seo_data\n',
    'if data:\n',
    '    st.subheader("SEO-Friendly Tags")\n',
    '    cols = st.columns(3)\n',
    '    for i, tag in enumerate(data["tags"]):\n',
    '        with cols[i % 3]:\n',
    '            st.write(f"#{tag}")\n',
    '    if st.button("Copy Tags"):\n',
    '        st.code(" ".join(f"#{t}" for t in data["tags"]))\n',
    '    st.divider()\n',
    '    st.subheader("Target Audience")\n',
    '    st.write(data["audience"])\n',
    '    st.divider()\n',
    '    st.subheader("Timestamps")\n',
    '    for ts in data["timestamps"]:\n',
    '        st.markdown(f"**{ts[chr(39)]time[chr(39)]}** - {ts[chr(39)]description[chr(39)]}")\n',
    '    st.divider()\n',
    '    st.subheader("Flaws & Improvements")\n',
    '    for flaw in data["flaws"]:\n',
    '        st.markdown(f"**Issue:** {flaw[chr(39)]issue[chr(39)]}\\n\\n**Why:** {flaw[chr(39)]why_it_hurts[chr(39)]}\\n\\n**Fix:** {flaw[chr(39)]fix[chr(39)]}")\n',
]

# Write app.py lines to disk
with open('app.py', 'w') as fh:
    fh.writelines(APP_LINES)
print('app.py written to disk.')

# ── Launch Streamlit as background process ────────────────────────────────────
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')

if not NGROK_TOKEN:
    print('NGROK_AUTHTOKEN not found in Colab Secrets.')
    print('Add it at ngrok.com -> Your Authtoken, then re-run this cell.')
else:
    ngrok.set_auth_token(NGROK_TOKEN)

    # Start Streamlit in the background
    proc = subprocess.Popen(
        ['streamlit', 'run', 'app.py',
         '--server.port=8501',
         '--server.headless=true',
         '--server.enableCORS=false'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    time.sleep(4)   # allow Streamlit to fully start up

    # Create a public ngrok tunnel to local port 8501
    public_url = ngrok.connect(8501)
    print(f'\nStreamlit app live at: {public_url}')
    print('Open that URL in your browser.')


## Cell 12 - Zip and Download All Outputs
Compresses everything in `OUTPUT_DIR` (JSON, tags, timestamps, flaws,
MLflow runs) into a single archive and triggers a browser download.

In [ ]:
import shutil
from google.colab import files  # type: ignore

# ── Archive path (shutil.make_archive appends .zip automatically) ─────────────
ZIP_BASE = '/content/yt_seo_outputs'
ZIP_FILE = ZIP_BASE + '.zip'

# ── Create the zip from the OUTPUT_DIR tree ───────────────────────────────────
shutil.make_archive(
    base_name=ZIP_BASE,
    format='zip',
    root_dir=OUTPUT_DIR,
)

print(f'Archive created: {ZIP_FILE}')

# ── Trigger browser download ──────────────────────────────────────────────────
files.download(ZIP_FILE)
print('Download triggered. Check your browser download bar.')
